# Séance 14 — Tableau de bord Power BI et IA générative pour le triage
*Session 14 — Power BI dashboard and generative AI for alert triage*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/14_bi_ia_generative.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ● séance complète |

**À la fin de cette séance, vous saurez :**

1. décrire un modèle en étoile (faits, dimensions) pour un tableau de bord de fraude ;
2. calculer correctement des indicateurs agrégés (ratio de sommes, pas moyenne de ratios) ;
3. construire une recherche documentaire (RAG) par TF-IDF sur des politiques internes ;
4. évaluer un assistant de triage : garde-fous, vérité terrain, juge LLM.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("exporter-bi")
dossier = cfg.chemin("exports_bi")
for f in sorted(dossier.glob("*.csv")):
    print(f"{f.name:<34} {sum(1 for _ in open(f, encoding='utf-8-sig')) - 1:>6} lignes")

**Modèle en étoile** (fichier `powerbi/Fraude360.pbip`) : tables de **faits** (`fait_kpi_quotidien`, `fait_alertes`,
`fait_typologie_quotidien`, `fait_derive`, `fait_ab_test`…) reliées aux **dimensions** `dim_date` et `dim_typologie`.
Les mesures DAX sont des ratios de sommes, par ex. `Taux de détection = DIVIDE(SUM(fraudes_detectees), SUM(fraudes))`.

In [ ]:
kpi = pd.read_csv(dossier / "fait_kpi_quotidien.csv", parse_dates=["date"])
kpi.head()

### Exercice 14.1 — 🟢 Tous les parcours

Calculez `taux_global`, le taux de détection de toute la période (somme des fraudes détectées / somme des fraudes),
et `moyenne_des_taux`, la moyenne des taux quotidiens. Lequel faut-il afficher dans Power BI ? Calculez aussi `cout_par_1000`,
le coût total par 1 000 transactions.

In [ ]:
# VOTRE CODE ICI
taux_global, moyenne_des_taux, cout_par_1000 = ..., ..., ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier_proche(taux_global, kpi.fraudes_detectees.sum() / kpi.fraudes.sum(), 1e-12)
verifier_proche(cout_par_1000, 1000 * kpi.cout_total.sum() / kpi.transactions.sum(), 1e-9)

### Exercice 14.2 — 🟣 Gestion + Informatique

Joignez `fait_typologie_quotidien.csv` à `dim_typologie.csv` (clé : `typologie_fraude` ↔ `typologie`) et calculez
`par_connaissance` : taux de détection (somme `detectes` / somme `cas`) selon `connue_du_modele_v1`.

In [ ]:
# VOTRE CODE ICI
par_connaissance = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(len(par_connaissance) == 2, "Deux groupes : typologies connues et inconnues du modèle v1")
verifier(par_connaissance.max() > par_connaissance.min(), "Les typologies connues sont mieux détectées")

## IA générative : triage des alertes avec recherche documentaire (RAG)

Pour chaque alerte bloquée, un modèle de langage **local** (Ollama, rien ne sort du poste — Loi 25) reçoit l'alerte et les
3 passages de politique les plus pertinents, puis propose une action (`BLOQUER`, `REVUE_MANUELLE`, `MISE_EN_ATTENTE`) en citant ses
sources. Sans Ollama, un client déterministe de repli produit des sorties au même format : le pipeline reste reproductible.

In [ ]:
from fraude360.genai import politiques
index = politiques.Index()
for p, sim in index.rechercher("virement vers un nouveau bénéficiaire, montant élevé, la nuit"):
    print(f"{sim:.3f}  [{p.identifiant}] {p.titre}")
notes = pd.read_csv(cfg.chemin("artefacts") / "genai" / "notes_triage.csv")
notes[["transaction_id", "typologie", "action", "confiance", "politiques_citees", "est_fraude"]].head()

### Exercice 14.3 — 🔷 Informatique — approfondissement

Reconstruisez la recherche : un `TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)` ajusté sur
`f"{p.titre}. {p.texte}"` pour chaque passage de `politiques.CORPUS`, puis `rechercher(requete, k=3)` qui renvoie la liste des
`identifiant` des k passages les plus similaires (similarité cosinus > 0), du plus au moins similaire.

In [ ]:
# VOTRE CODE ICI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
def rechercher(requete, k=3):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
for _q in ["micro-montants répétés en ligne", "virement nouveau bénéficiaire", "carte volée sans contact"]:
    verifier(rechercher(_q) == [p.identifiant for p, _ in index.rechercher(_q)], f"Même résultat pour « {_q} »")

In [ ]:
import json
rt = lire_json(cfg.chemin("artefacts") / "genai" / "rapport_triage.json")
print(json.dumps(rt, indent=1, ensure_ascii=False)[:1800])

**❓ Question (🔶 Gestion — décision d'affaires) —** Quels risques et quelles conditions d'utilisation mettriez-vous dans la politique d'usage de cet assistant ? (hallucination, explicabilité, renseignements personnels, décision humaine finale)

*Votre réponse ici.*

## Bilan

- Un tableau de bord fiable repose sur un modèle en étoile et des mesures bien définies (ratio de sommes).
- Le RAG ancre les réponses d'un modèle de langage dans des politiques internes vérifiables.
- Un assistant génératif s'évalue : format (garde-fous), utilité (vérité terrain), qualité (juge).

**Pour aller plus loin :**

- `powerbi/` (projet .pbip) et `src/fraude360/genai/` (triage, évaluation, client Ollama).

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*